In [ ]:
!pip install -q kagglehub librosa
import kagglehub, glob, os, collections, librosa, numpy as np

In [ ]:
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Downloaded to:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Downloaded to: /kaggle/input/gtzan-dataset-music-genre-classification


In [ ]:
import glob, os, collections
data = sorted(glob.glob(os.path.join(path, "**", "*.wav"), recursive=True))
counts = collections.Counter(os.path.basename(os.path.dirname(i)) for i in data)
print(f"Found {len(data)} data files across {len(counts)} genres\n")
for genre, n in sorted(counts.items()):
    print(f"  {genre:10s} {n}")

Found 1000 data files across 10 genres

  blues      100
  classical  100
  country    100
  disco      100
  hiphop     100
  jazz       100
  metal      100
  pop        100
  reggae     100
  rock       100


In [ ]:
from sklearn.model_selection import train_test_split

genres = sorted(set(os.path.basename(os.path.dirname(i)) for i in data))
genre_index = {g: i for i, g in enumerate(genres)}

labels = [genre_index[os.path.basename(os.path.dirname(i))] for i in data]

train_data, temp_data, train_labels, temp_labels = train_test_split(
    data, labels, test_size=0.30, stratify=labels, random_state=42)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_data)} songs")
print(f"Val:   {len(val_data)} songs")
print(f"Test:  {len(test_data)} songs")
print("Genre index map:", genre_index)

Train: 700 songs
Val:   150 songs
Test:  150 songs
Genre index map: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [ ]:
import torch
import numpy as np
import librosa
from torch.utils.data import Dataset, DataLoader

SAMPLE_RATE = 22050      # samples per second we resample every clip to
CHUNK_SECONDS = 5        # length of each chunk
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS
N_MELS = 128             # height of the spectrogram

class GTZANDataset(Dataset):
    def __init__(self, data, labels):
        self.index = []
        for filepath, label in zip(data, labels):
            try:
                duration = librosa.get_duration(path=filepath)
            except Exception:
                continue
            n_chunks = int(duration // CHUNK_SECONDS)
            for c in range(n_chunks):
                self.index.append((filepath, label, c))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        filepath, label, chunk = self.index[i]

        offset = chunk * CHUNK_SECONDS
        y, sr = librosa.load(filepath, sr=SAMPLE_RATE,
                             offset=offset, duration=CHUNK_SECONDS)

        if len(y) < CHUNK_SAMPLES:
            y = np.pad(y, (0, CHUNK_SAMPLES - len(y)))

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)
        return tensor, label

In [ ]:
train_ds = GTZANDataset(train_data, train_labels)
val_ds   = GTZANDataset(val_data,   val_labels)
test_ds  = GTZANDataset(test_data,  test_labels)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

print(f"Train chunks: {len(train_ds)}")
print(f"Val chunks:   {len(val_ds)}")
print(f"Test chunks:  {len(test_ds)}")

/tmp/ipykernel_6683/1633446368.py:16: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  duration = librosa.get_duration(path=filepath)


Train chunks: 4185
Val chunks:   900
Test chunks:  900


In [ ]:
import torch.nn as nn
class GenreCNN(nn.Module):
  def __init__(self, n_classes=10):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),

    )

    self.pool = nn.AdaptiveAvgPool2d(1)

    self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
    )

  def forward(self, x):
    x = self.conv(x)
    x = self.pool(x)
    x = self.head(x)
    return x
